# AnimalPoseTracker Demo

Run a 10-epoch AnimalRTPose-N workflow on Kaggle: training, validation, test prediction from `.pt`, ONNX export, and test prediction from ONNX. Attach the dataset under Kaggle input `AnimalDatasets`; it is read directly from `/kaggle/input` without copying. Outputs are saved under `/kaggle/working/wux024`. Enable a GPU accelerator before running.

Set `RUN_MODE` to `new` or `load`. In load mode, attach the trained checkpoint as a Kaggle input and set `CHECKPOINT_PATH` to its `/kaggle/input/.../best.pt` path.

The Kaggle runtime is a cloud Linux machine. Repository update, installation, training, validation, prediction, and export run in Bash command cells. Project creation and configuration use `animalpose-cli create`. Run cells from the top; the repository cell updates `preview` and prints the commit used.

In [ ]:
# Run this cell, choose a mode and checkpoint path, then click Apply.
# If the controls do not render in this Kaggle session, edit the fallback values below.
RUN_MODE = "new"
CHECKPOINT_PATH = ""

try:
    import ipywidgets as widgets
    from IPython.display import display

    mode_widget = widgets.Dropdown(
        options=[("New training project", "new"), ("Load checkpoint", "load")],
        value=RUN_MODE,
        description="Mode:",
        layout=widgets.Layout(width="360px"),
    )
    checkpoint_widget = widgets.Text(
        value=CHECKPOINT_PATH,
        description="Checkpoint:",
        placeholder="/kaggle/input/.../best.pt",
        layout=widgets.Layout(width="760px"),
    )
    apply_button = widgets.Button(description="Apply", button_style="primary")
    status = widgets.HTML()

    def apply_run_options(_=None):
        global RUN_MODE, CHECKPOINT_PATH
        RUN_MODE = mode_widget.value
        CHECKPOINT_PATH = checkpoint_widget.value.strip()
        status.value = f"<b>Applied:</b> {RUN_MODE} — continue with the next cell."
        if RUN_MODE == "load":
            status.value += " — checkpoint path set" if CHECKPOINT_PATH else " — enter a checkpoint path"

    apply_button.on_click(apply_run_options)
    display(widgets.VBox([mode_widget, checkpoint_widget, apply_button, status]))
    apply_run_options()
except ImportError:
    print("ipywidgets unavailable; use the RUN_MODE and CHECKPOINT_PATH fallback values above.")


In [ ]:
from datetime import datetime
from pathlib import Path

REPO_URL = "https://github.com/wux024/AnimalPoseTracker.git"
BRANCH = "preview"
REPO_DIR = Path("/kaggle/working/wux024/AnimalPoseTracker")
PROJECTS_ROOT = REPO_DIR.parent
ANIMAL_DATASETS = Path("/kaggle/input")
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
PREDICT_SPLIT = "test"
EXPORT_DIR = f"runs/export-{RUN_ID}"
EPOCHS, BATCH_SIZE, IMAGE_SIZE, NUM_WORKERS, DEVICE = 10, 4, 640, 2, "0"

DATA_SOURCE_ROOT = next(
    images.parent
    for images in ANIMAL_DATASETS.rglob("images")
    if "trimouse" in str(images.parent).lower()
    and all((images / split).is_dir() for split in ("train", "val", "test"))
)

if RUN_MODE == "load":
    BEST_CHECKPOINT = Path(CHECKPOINT_PATH).expanduser().resolve()
else:
    BEST_CHECKPOINT = PROJECTS_ROOT / f"trimouse-kaggle-AnimalRTPose-N-{RUN_ID}" / "runs/train/weights/best.pt"

REPO_DIR_STR = str(REPO_DIR)
PROJECT_DIR = PROJECTS_ROOT / f"trimouse-kaggle-AnimalRTPose-N-{RUN_ID}"
PROJECT_DIR_STR = str(PROJECT_DIR)
%env REPO_URL={REPO_URL}
%env BRANCH={BRANCH}
%env REPO_DIR={REPO_DIR_STR}
%env PROJECTS_ROOT={PROJECTS_ROOT}
%env DATA_SOURCE_ROOT={DATA_SOURCE_ROOT}
%env RUN_ID={RUN_ID}
%env PROJECT_DIR={PROJECT_DIR_STR}
%env BEST_CHECKPOINT={BEST_CHECKPOINT}
%env RUN_MODE={RUN_MODE}
%env PREDICT_SPLIT={PREDICT_SPLIT}
%env EXPORT_DIR={EXPORT_DIR}
%env EPOCHS={EPOCHS}
%env BATCH_SIZE={BATCH_SIZE}
%env IMAGE_SIZE={IMAGE_SIZE}
%env NUM_WORKERS={NUM_WORKERS}
%env DEVICE={DEVICE}


In [ ]:
%%bash
set -e
mkdir -p "$(dirname "$REPO_DIR")"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
else
  git -C "$REPO_DIR" pull --ff-only origin "$BRANCH"
fi
git -C "$REPO_DIR" log -1 --oneline


In [ ]:
%%bash
set -e
python -m pip install -e "$REPO_DIR"
python -m pip install pycocotools onnx onnxsim onnxruntime


In [ ]:
%%bash
set -e
cd "$REPO_DIR"
if [ "$RUN_MODE" = "new" ]; then
  animalpose-cli create \
    --dataset animalposetracker/cfg/datasets/trimouse.yaml \
    --dataset-root "$DATA_SOURCE_ROOT" \
    --workspace "$PROJECTS_ROOT" \
    --name trimouse --worker kaggle --model AnimalRTPose --scale N \
    --date "$RUN_ID" --epochs "$EPOCHS" --batch "$BATCH_SIZE" \
    --imgsz "$IMAGE_SIZE" --workers "$NUM_WORKERS" --device "$DEVICE"
else
  animalpose-cli create \
    --dataset animalposetracker/cfg/datasets/trimouse.yaml \
    --dataset-root "$DATA_SOURCE_ROOT" \
    --workspace "$PROJECTS_ROOT" \
    --name trimouse --worker kaggle --model AnimalRTPose --scale N \
    --date "$RUN_ID" --epochs "$EPOCHS" --batch "$BATCH_SIZE" \
    --imgsz "$IMAGE_SIZE" --workers "$NUM_WORKERS" --device "$DEVICE" \
    --no-pretrained
fi


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
if [ "$RUN_MODE" = "new" ]; then
  animalpose-cli train --config configs/other.yaml
fi


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli val --config configs/other.yaml --weights "$BEST_CHECKPOINT"


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli predict --config configs/other.yaml --weights "$BEST_CHECKPOINT" --split "$PREDICT_SPLIT"


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli export --config configs/other.yaml --weights "$BEST_CHECKPOINT" --format onnx --output-dir "$EXPORT_DIR"


In [ ]:
%%bash
set -e
cd "$PROJECT_DIR"
animalpose-cli predict --config configs/other.yaml --weights "$PROJECT_DIR/$EXPORT_DIR/best.onnx" --split "$PREDICT_SPLIT"
